# 🚀 Private RAG Stack with EmbeddingGemma & SQLite-vec

## 🔒 100% Private | 💰 Zero Cost | 📱 Offline Capable

This notebook demonstrates how to build a complete RAG (Retrieval Augmented Generation) system using:

- **EmbeddingGemma**: Google's efficient 300M parameter embedding model (via Ollama)
- **SQLite-vec**: Fast vector similarity search in SQLite
- **Qwen3**: Efficient local language model via Ollama

### What You'll Learn:
1. How to scrape and prepare documentation
2. Generate embeddings with EmbeddingGemma (via Ollama)
3. Store vectors in SQLite with sqlite-vec
4. Perform semantic search
5. Generate contextual responses with local LLM

## 📚 Step 1: Import Required Libraries

We'll use a minimal set of libraries for our RAG pipeline:

In [1]:
import sqlite3
import sqlite_vec
import ollama
import requests
from bs4 import BeautifulSoup
import struct
import time
import os
import json
import numpy as np

## ⚙️ Step 2: Configuration & Helper Functions

Let's set up our configuration and utility functions:

In [2]:
# Configuration - Load from .env file
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configuration from .env
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'embeddinggemma:latest')  # Local Ollama embedding model
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-oss:20b')  # Local Ollama LLM model
EMBEDDING_DIMS = int(os.getenv('EMBEDDING_DIMS', '768'))  # 256 for 3x speed, 768 for max quality
DB_FILE = "rag_vectors.db"
TABLE_NAME = "documents"
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')

# Vector serialization helper
def serialize_f32(vector):
    """Convert float vector to bytes for SQLite storage"""
    return struct.pack("%sf" % len(vector), *vector)

# Ollama embedding function
def get_ollama_embedding(text, model=EMBEDDING_MODEL):
    """Get embeddings from Ollama embedding model"""
    try:
        response = ollama.embeddings(
            model=model,
            prompt=text,
            options={'truncate_dim': EMBEDDING_DIMS}
        )
        return response['embedding']
    except Exception as e:
        print(f"Error getting embedding: {e}")
        return None

print("✅ Configuration loaded!")

✅ Configuration loaded!


## 🌐 Step 3: Scrape Documentation

We'll scrape official documentation from key sources to build our knowledge base:

In [3]:
# Documentation sources for our RAG system
docs_to_scrape = {
    'sqlite_vec_python': 'https://alexgarcia.xyz/sqlite-vec/python.html',
    'sqlite_vec_demo': 'https://raw.githubusercontent.com/asg017/sqlite-vec/main/examples/simple-python/demo.py',
    'embeddinggemma_google_blog': 'https://developers.googleblog.com/en/introducing-embeddinggemma/',
    'huggingface_embeddinggemma': 'https://huggingface.co/google/embeddinggemma-300m',
    'huggingface_embeddinggemma_blog': 'https://huggingface.co/blog/embeddinggemma',
    'qwen3_ollama': 'https://ollama.com/library/qwen3',
    'sentence_transformers': 'https://sbert.net/docs/package_reference/sentence_transformer/SentenceTransformer.html'
}

headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

# Create docs directory
os.makedirs('docs', exist_ok=True)

print("📥 Scraping documentation...")
scraped_docs = []

for name, url in docs_to_scrape.items():
    try:
        print(f"📄 Fetching: {name}")
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        
        # Parse and extract text
        soup = BeautifulSoup(response.content, 'html.parser')
        for element in soup(['script', 'style']):
            element.decompose()
        
        text_content = soup.get_text(separator='\n', strip=True)
        
        if text_content.strip():
            # Save locally
            filepath = f"docs/{name}.txt"
            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(f"Source: {url}\n")
                f.write("=" * 80 + "\n\n")
                f.write(text_content)
            
            scraped_docs.append((name, text_content))
            print(f"   ✅ Saved: {name} ({len(text_content)} chars)")
        
        time.sleep(1)  # Be respectful to servers
        
    except Exception as e:
        print(f"   ❌ Error fetching {name}: {e}")

print(f"\n📊 Successfully scraped {len(scraped_docs)} documents")

📥 Scraping documentation...
📄 Fetching: sqlite_vec_python
   ✅ Saved: sqlite_vec_python (4801 chars)
📄 Fetching: sqlite_vec_demo
   ✅ Saved: sqlite_vec_demo (1213 chars)
📄 Fetching: embeddinggemma_google_blog
   ✅ Saved: embeddinggemma_google_blog (9550 chars)
📄 Fetching: huggingface_embeddinggemma
   ✅ Saved: huggingface_embeddinggemma (15408 chars)
📄 Fetching: huggingface_embeddinggemma_blog
   ✅ Saved: huggingface_embeddinggemma_blog (49446 chars)
📄 Fetching: qwen3_ollama
   ✅ Saved: qwen3_ollama (4967 chars)
📄 Fetching: sentence_transformers
   ✅ Saved: sentence_transformers (80210 chars)

📊 Successfully scraped 7 documents


## 🧠 Step 4: Initialize EmbeddingGemma Model (via Ollama)

Load EmbeddingGemma via Ollama for generating high-quality embeddings:

> **Note**: Make sure you have Ollama installed and the embedding model downloaded: `ollama pull embeddinggemma:latest`

In [4]:
# Test Ollama connection and embedding model
print(f"🤖 Testing Ollama connection and {EMBEDDING_MODEL}...")

# Test the model with a sample
sample_text = "EmbeddingGemma is Google's efficient embedding model"
sample_embedding = get_ollama_embedding(sample_text)

if sample_embedding:
    print(f"✅ {EMBEDDING_MODEL} loaded successfully!")
    print(f"Sample embedding: {sample_embedding[:10]}... (showing first 10 values)")
    print(f"📊 Sample embedding length: {len(sample_embedding)}")
    print(f"💡 Using {EMBEDDING_DIMS} dimensions for embeddings")
else:
    print("❌ Failed to load embedding model. Check Ollama installation and model availability.")

🤖 Testing Ollama connection and embeddinggemma:latest...
✅ embeddinggemma:latest loaded successfully!
Sample embedding: [-0.11867867410182953, -0.041302602738142014, 0.024900641292333603, 0.02035672403872013, 0.051944755017757416, 0.06384780257940292, -0.0016943488735705614, 0.050311461091041565, 0.04542052000761032, -0.021916549652814865]... (showing first 10 values)
📊 Sample embedding length: 768
💡 Using 768 dimensions for embeddings


## 🗄️ Step 5: Setup SQLite Vector Database

Initialize SQLite with the sqlite-vec extension for efficient vector storage and similarity search:

In [5]:
# Initialize SQLite with vector extension
print("🗄️ Setting up vector database...")
conn = sqlite3.connect(DB_FILE)
conn.enable_load_extension(True)
sqlite_vec.load(conn)
conn.enable_load_extension(False)

# Create vector table - FIXED VERSION (single line f-string)
create_table_sql = f"CREATE VIRTUAL TABLE IF NOT EXISTS {TABLE_NAME} USING vec0(text TEXT, source TEXT, embedding float[{EMBEDDING_DIMS}])"
conn.execute(create_table_sql)
conn.commit()
print("✅ Vector database ready!")

🗄️ Setting up vector database...
✅ Vector database ready!


## 🔄 Step 6: Smart Token-Based Chunking & Embeddings

Process our scraped documents using intelligent token-based chunking for optimal embedding quality:

**Why Token-Based Chunking?**
- Uses the **same tokenizer** as EmbeddingGemma for perfect alignment
- **Respects token boundaries** instead of arbitrary character limits
- **Prevents word splitting** that degrades embedding quality
- **Consistent chunk sizes** measured in actual tokens, not characters

In [6]:
def token_based_chunking(text, max_tokens=2048, overlap_tokens=100):
    """
    Token-based chunking using Ollama's tokenizer.
    Much more accurate than character-based chunking for optimal embeddings.
    """
    # Simple token-based chunking using word count as proxy
    words = text.split()
    
    if len(words) <= max_tokens:
        return [text]  # No need to chunk
    
    chunks = []
    start = 0
    
    while start < len(words):
        # Get chunk words
        end = min(start + max_tokens, len(words))
        chunk_words = words[start:end]
        
        # Join back to text
        chunk_text = ' '.join(chunk_words)
        chunks.append(chunk_text.strip())
        
        # Move start position with overlap
        if end >= len(words):
            break
        start = end - overlap_tokens
    
    return chunks

# Process all documents with intelligent token-based chunking
print("📝 Chunking documents with token-based precision...")
all_chunks = []
all_sources = []
all_embeddings = []

for source_name, content in scraped_docs:
    # Use token-based chunking
    chunks = token_based_chunking(content, max_tokens=2048, overlap_tokens=100)
    print(f"📄 {source_name}: {len(chunks)} chunks")
    
    # Generate embeddings for all chunks using Ollama
    chunk_embeddings = []
    for chunk in chunks:
        embedding = get_ollama_embedding(chunk)
        if embedding:
            chunk_embeddings.append(embedding)
        else:
            print(f"   ⚠️  Failed to get embedding for chunk")
    
    all_chunks.extend(chunks)
    all_sources.extend([source_name] * len(chunks))
    all_embeddings.extend(chunk_embeddings)

print(f"\n📊 Total chunks created: {len(all_chunks)}")
print(f"🧮 Total embeddings generated: {len(all_embeddings)}")
print(f"💡 Using token-based chunking ensures optimal embedding quality!")

📝 Chunking documents with token-based precision...
📄 sqlite_vec_python: 1 chunks
📄 sqlite_vec_demo: 1 chunks
📄 embeddinggemma_google_blog: 1 chunks
📄 huggingface_embeddinggemma: 2 chunks
📄 huggingface_embeddinggemma_blog: 4 chunks
📄 qwen3_ollama: 1 chunks
📄 sentence_transformers: 7 chunks

📊 Total chunks created: 17
🧮 Total embeddings generated: 17
💡 Using token-based chunking ensures optimal embedding quality!


## 💾 Step 7: Store Embeddings in Vector Database

Insert all our document chunks and their embeddings into the SQLite vector database:

In [7]:
# Store all chunks and embeddings
print("💾 Storing embeddings in vector database...")

for i, (chunk, source, embedding) in enumerate(zip(all_chunks, all_sources, all_embeddings)):
    conn.execute(f"""
        INSERT INTO {TABLE_NAME} (rowid, text, source, embedding)
        VALUES (?, ?, ?, ?)
    """, (i + 1, chunk, source, serialize_f32(embedding)))
    
    if (i + 1) % 10 == 0:
        print(f"🔄 Processed {i + 1}/{len(all_chunks)} chunks...")

conn.commit()
print("\n✅ All embeddings stored successfully!")

# Verify our data
cursor = conn.execute(f"SELECT COUNT(*) FROM {TABLE_NAME}")
count = cursor.fetchone()[0]
print(f"📊 Database contains {count} documents ready for search")

💾 Storing embeddings in vector database...
🔄 Processed 10/17 chunks...

✅ All embeddings stored successfully!
📊 Database contains 17 documents ready for search


## 🔍 Step 8: Semantic Search Function

Create a function to perform semantic search using our vector database:

In [8]:
def semantic_search(query_text, top_k=3):
    """Perform semantic search and return relevant documents"""
    
    # Generate query embedding using Ollama
    query_embedding = get_ollama_embedding(query_text)
    
    if not query_embedding:
        print("❌ Failed to get query embedding")
        return []
    
    # Search for similar documents
    cursor = conn.execute(f"""
        SELECT rowid, text, source, distance
        FROM {TABLE_NAME}
        WHERE embedding MATCH ?
        ORDER BY distance
        LIMIT ?
    """, (serialize_f32(query_embedding), top_k))
    
    results = cursor.fetchall()
    
    print(f"🔍 Found {len(results)} relevant documents:")
    contexts = []
    
    for rowid, text, source, distance in results:
        contexts.append(text)
        print(f"📄 Source: {source} | Distance: {distance:.4f}")
        print(f"📝 Preview: {text[:100]}...\n")
    
    return contexts

# Test semantic search
test_query = "How does EmbeddingGemma work?"
print(f"🧪 Testing search with query: '{test_query}'")
test_results = semantic_search(test_query)
print(f"✅ Search test completed!")

🧪 Testing search with query: 'How does EmbeddingGemma work?'
🔍 Found 3 relevant documents:
📄 Source: embeddinggemma_google_blog | Distance: 0.9861
📝 Preview: Introducing EmbeddingGemma: The Best-in-Class Open Model for On-Device Embeddings
            
     ...

📄 Source: huggingface_embeddinggemma | Distance: 1.0453
📝 Preview: google/embeddinggemma-300m · Hugging Face Hugging Face Models Datasets Spaces Community Docs Enterpr...

📄 Source: huggingface_embeddinggemma | Distance: 1.0478
📝 Preview: exploration of de-biasing techniques during model training, fine-tuning, and other use cases. Misuse...

✅ Search test completed!


## 🤖 Step 9: RAG Query Function with Local LLM

Combine semantic search with local LLM to generate contextual responses:

> **Note**: Make sure you have Ollama installed and the Qwen3 model downloaded: `ollama pull qwen3:4b`

In [9]:
def rag_query(question, top_k=3):
    """Complete RAG pipeline: search + generate response"""
    
    print(f"❓ Question: {question}")
    print("=" * 60)
    
    # Step 1: Semantic search
    contexts = semantic_search(question, top_k)
    
    if not contexts:
        return "❌ No relevant information found."
    
    # Step 2: Build prompt with context
    combined_context = "\n\n".join(contexts)
    prompt = f"""Use the following contexts to answer the question comprehensively.
If you don't know the answer based on the provided contexts, just say that you don't know.

Contexts:
{combined_context}

Question: {question}

Answer:"""
    
    # Step 3: Generate response with local LLM
    print(f"🤖 Generating response with {LLM_MODEL}...\n")
    
    try:
        # Stream response for real-time output
        stream = ollama.chat(
            model=LLM_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            stream=True
        )
        
        response = ""
        for chunk in stream:
            if 'message' in chunk and 'content' in chunk['message']:
                content = chunk['message']['content']
                print(content, end='', flush=True)
                response += content
        
        print("\n" + "=" * 60)
        return response
        
    except Exception as e:
        error_msg = f"❌ Error with LLM: {e}"
        print(error_msg)
        return error_msg

print("✅ RAG query function ready!")

✅ RAG query function ready!


## 🎯 Step 10: Demo Queries - Let's Test Our RAG System!

Now let's put our RAG system to work with some interesting questions:

In [10]:
# Demo Question 1: About EmbeddingGemma
response1 = rag_query("What makes EmbeddingGemma special for mobile applications?")

❓ Question: What makes EmbeddingGemma special for mobile applications?
🔍 Found 3 relevant documents:
📄 Source: embeddinggemma_google_blog | Distance: 1.0455
📝 Preview: Introducing EmbeddingGemma: The Best-in-Class Open Model for On-Device Embeddings
            
     ...

📄 Source: huggingface_embeddinggemma | Distance: 1.1000
📝 Preview: exploration of de-biasing techniques during model training, fine-tuning, and other use cases. Misuse...

📄 Source: huggingface_embeddinggemma | Distance: 1.1525
📝 Preview: google/embeddinggemma-300m · Hugging Face Hugging Face Models Datasets Spaces Community Docs Enterpr...

🤖 Generating response with gpt-oss:20b...

**EmbeddingGemma is engineered specifically for mobile‑first, on‑device AI – here’s why it stands out for mobile applications:**

| Feature | Why it matters for mobile |
|---------|---------------------------|
| **Ultra‑compact size (≈ 308 M parameters)** | Fits comfortably on modern phones, tablets and edge devices; requires only ~200 MB

In [11]:
# Demo Question 2: About SQLite-vec
response2 = rag_query("How do I use SQLite-vec with Python?")

❓ Question: How do I use SQLite-vec with Python?
🔍 Found 3 relevant documents:
📄 Source: sqlite_vec_python | Distance: 0.7670
📝 Preview: sqlite-vec in Python | sqlite-vec
🚧🚧🚧 This documentation is a work-in-progress! 🚧🚧🚧
Skip to content
...

📄 Source: sqlite_vec_demo | Distance: 0.9585
📝 Preview: import sqlite3
import sqlite_vec

from typing import List
import struct


def serialize_f32(vector: ...

📄 Source: sentence_transformers | Distance: 1.1365
📝 Preview: SentenceTransformer — Sentence Transformers documentation Getting Started Installation Install with ...

🤖 Generating response with gpt-oss:20b...

**Using SQLite‑vec from Python – a quick‑start guide**

SQLite‑vec is a SQLite extension that gives you fast vector‐operations (length, distance, KNN queries, etc.) and a virtual table (`vec0`) for storing dense embeddings. The Python side is intentionally thin – you only need to load the native extension and then use normal `sqlite3` code.

Below is a “recipe” that covers everything 

In [12]:
# Demo Question 3: About Qwen3
response3 = rag_query("What are the key features of Qwen3 model?")

❓ Question: What are the key features of Qwen3 model?
🔍 Found 3 relevant documents:
📄 Source: qwen3_ollama | Distance: 0.8717
📝 Preview: qwen3
Models
GitHub
Discord
Docs
Cloud
Sign in
Download
Models
Download
GitHub
Discord
Docs
Cloud
Si...

📄 Source: huggingface_embeddinggemma | Distance: 1.0846
📝 Preview: google/embeddinggemma-300m · Hugging Face Hugging Face Models Datasets Spaces Community Docs Enterpr...

📄 Source: sentence_transformers | Distance: 1.0923
📝 Preview: int = 500, checkpoint_save_total_limit: int = 0, resume_from_checkpoint: bool = False ) → None [sour...

🤖 Generating response with gpt-oss:20b...

**Key features of the Qwen 3 family**

| Feature | What it means | Why it matters |
|---------|---------------|----------------|
| **Latest generation in the Qwen series** | Qwen 3 builds on earlier Qwen models with new architectures and training data. | It brings the most up‑to‑date capabilities and performance gains. |
| **Dense & Mixture‑of‑Experts (MoE) models** | Provi

In [13]:
# Demo Question 4: Technical concepts
response4 = rag_query("How does vector similarity search work?")

❓ Question: How does vector similarity search work?
🔍 Found 3 relevant documents:
📄 Source: sqlite_vec_demo | Distance: 1.0842
📝 Preview: import sqlite3
import sqlite_vec

from typing import List
import struct


def serialize_f32(vector: ...

📄 Source: sqlite_vec_python | Distance: 1.0971
📝 Preview: sqlite-vec in Python | sqlite-vec
🚧🚧🚧 This documentation is a work-in-progress! 🚧🚧🚧
Skip to content
...

📄 Source: sentence_transformers | Distance: 1.1199
📝 Preview: SentenceTransformer — Sentence Transformers documentation Getting Started Installation Install with ...

🤖 Generating response with gpt-oss:20b...

**Vector similarity search** is the process of finding the vectors in a database that are *closest* to a given query vector.  
In the context of **sqlite‑vec** (the example code in the context) it works as follows:

| Step | What happens | Why it matters |
|------|--------------|----------------|
| **1. Vector format** | Vectors are stored as a binary blob.  In Python you convert a

## 🎉 Congratulations!

You've successfully built a complete private RAG system! Here's what we accomplished:

### ✅ What We Built:
- **Document Scraping**: Automated collection from web sources
- **Smart Chunking**: Optimized text segmentation for better retrieval
- **Modern Embeddings**: Google's EmbeddingGemma with Matryoshka learning (via Ollama)
- **Vector Database**: SQLite-vec for fast similarity search
- **Local LLM**: Qwen3 for generating contextual responses
- **Complete Privacy**: Everything runs locally, no API calls

### 🚀 Key Benefits:
- **100% Private**: All processing happens on your machine
- **Zero Cost**: No API fees or usage limits
- **Offline Capable**: Works without internet after initial setup
- **Efficient**: EmbeddingGemma + SQLite-vec = fast performance
- **Scalable**: Can handle thousands of documents

### 🔄 Next Steps:
- Add more document sources to expand knowledge base
- Experiment with different chunking strategies
- Try other embedding dimensions (128, 512, 768)
- Implement conversation memory for multi-turn chats
- Build a simple web interface with Streamlit or Gradio

### 📚 Resources:
- [EmbeddingGemma Documentation](https://huggingface.co/google/embeddinggemma-300m)
- [SQLite-vec GitHub](https://github.com/asg017/sqlite-vec)
- [Ollama Models](https://ollama.com/library)

Happy building! 🎯

In [14]:
# Cleanup - close database connection
conn.close()
print("🧹 Database connection closed. Demo complete!")

🧹 Database connection closed. Demo complete!
